# Data-Driven Personalization in Retail
## A Machine Learning Approach to Enhancing Customer Lifetime Value
### Διπλωματική Εργασία — Παράρτημα Α: Πλήρης Κώδικας Ανάλυσης
**Φοιτήτρια:** Ελιάνα Παλόκα (Α.Μ. ΜΟΕΣ/2433)
**Επιβλέπων:** Χαράλαμπος Αγκυρόπουλος
**Πανεπιστήμιο Πειραιώς** | ΠΜΣ Οικονομική και Επιχειρησιακή Στρατηγική | 2025

---
### Οδηγίες Εκτέλεσης
1. Εκτελέστε **Κελί 1** (εγκατάσταση βιβλιοθηκών)
2. Εκτελέστε **Κελί 2** (ανέβασμα dataset Online Retail II .xlsx)
3. **Runtime → Run all** για εκτέλεση όλων με τη σειρά

**Dataset:** Online Retail II (Chen, 2015)
https://archive.ics.uci.edu/dataset/502/online+retail+ii

**Βιβλιοθήκες:** scikit-learn (Pedregosa et al., 2011), XGBoost (Chen & Guestrin, 2016),
LightGBM (Ke et al., 2017), TensorFlow (Abadi et al., 2016),
SHAP (Lundberg & Lee, 2017), lifetimes (Davidson-Pilon, 2019)

**Αναπαραγωγιμότητα:** random_state=42 σε όλα τα μοντέλα

In [ ]:
# ============================================================
# ΚΕΛΙ 1: Εγκατάσταση Βιβλιοθηκών
# ============================================================
!pip install lifetimes xgboost lightgbm shap --quiet
print('Βιβλιοθήκες εγκαταστάθηκαν')

In [ ]:
# ============================================================
# ΚΕΛΙ 2: Ανέβασμα Dataset
# ============================================================
from google.colab import files
uploaded = files.upload()
# Επιλέξτε: Copy of online_retail_II (version 1).xlsx

In [ ]:
# ============================================================
# ΚΕΛΙ 3: Imports
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                              silhouette_score, calinski_harabasz_score, davies_bouldin_score)
from scipy.stats import spearmanr
import xgboost as xgb
import lightgbm as lgb
import shap
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import summary_data_from_transaction_data

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
print('Imports OK')

In [ ]:
# ============================================================
# ΚΕΛΙ 4: Φόρτωση Dataset
# ============================================================
FILE = 'Copy of online_retail_II (version 1).xlsx'
print('Φορτώνω το Excel... (1-2 λεπτά)')
df1 = pd.read_excel(FILE, sheet_name='Year 2009-2010')
df2 = pd.read_excel(FILE, sheet_name='Year 2010-2011')
df_raw = pd.concat([df1, df2], ignore_index=True)
print(f'Αρχικές εγγραφές (raw): {len(df_raw):,}')
print(f'Στήλες: {list(df_raw.columns)}')

In [ ]:
# ============================================================
# ΚΕΛΙ 5: Καθαρισμός Δεδομένων
# Σημείωση: Το dataset είχε υποστεί προκαταρκτική επεξεργασία
# πριν την εισαγωγή του στον κώδικα. Τα βήματα καθαρισμού
# εφαρμόζονται για πληρότητα και τεκμηρίωση της διαδικασίας.
# ============================================================
df = df_raw[['Invoice','StockCode','Description','Quantity',
             'InvoiceDate','Price','Customer ID','Country']].copy()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue'] = df['Quantity'] * df['Price']

n0 = len(df)
# Βήμα 1: Αφαίρεση ακυρώσεων (Invoice που αρχίζει με 'C')
df = df[~df['Invoice'].astype(str).str.startswith('C')]
n1 = len(df)
# Βήμα 2: Αφαίρεση κενών Customer ID
df = df.dropna(subset=['Customer ID'])
n2 = len(df)
# Βήμα 3: Αφαίρεση αρνητικών/μηδενικών τιμών
df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]
n3 = len(df)
# Βήμα 4: Μετατροπή Customer ID σε integer
df['Customer ID'] = df['Customer ID'].astype(int)

print('=== Διαδικασία Καθαρισμού ===')
print(f'Αρχικές εγγραφές              : {n0:,}')
print(f'Μετά αφαίρεση ακυρώσεων       : {n1:,} (-{n0-n1:,})')
print(f'Μετά αφαίρεση κενών CustomerID: {n2:,} (-{n1-n2:,})')
print(f'Μετά αφαίρεση αρν. τιμών      : {n3:,} (-{n2-n3:,})')
print(f'Μοναδικοί πελάτες             : {df["Customer ID"].nunique():,}')
print(f'Χρονικό εύρος                 : {df["InvoiceDate"].min().date()} → {df["InvoiceDate"].max().date()}')

In [ ]:
# ============================================================
# ΚΕΛΙ 6: Temporal Split
# ============================================================
cutoff_date = pd.Timestamp('2011-06-01')
df_feat = df[df['InvoiceDate'] < cutoff_date].copy()
df_targ = df[df['InvoiceDate'] >= cutoff_date].copy()

print(f'Feature period: {df_feat["InvoiceDate"].min().date()} → {df_feat["InvoiceDate"].max().date()} ({len(df_feat):,} εγγραφές)')
print(f'Target period : {df_targ["InvoiceDate"].min().date()} → {df_targ["InvoiceDate"].max().date()} ({len(df_targ):,} εγγραφές)')

In [ ]:
# ============================================================
# ΚΕΛΙ 7: EDA — Μηνιαία Εξέλιξη Εσόδων (Διάγραμμα 4.1)
# ============================================================
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly = df.groupby('YearMonth')['Revenue'].sum()

plt.figure(figsize=(14,5))
plt.plot(monthly.index.astype(str), monthly.values, color='steelblue', linewidth=2, marker='o', markersize=4)
plt.title('Μηνιαία Εξέλιξη Εσόδων (2009-2011)', fontsize=14)
plt.xlabel('Μήνας'); plt.ylabel('Έσοδα (£)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('diagram_4_1_monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.1 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 8: EDA — Γεωγραφική Κατανομή (Διάγραμμα 4.2)
# ============================================================
country = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False).head(10)
country_pct = (country / country.sum() * 100).round(2)

plt.figure(figsize=(12,5))
bars = plt.bar(country_pct.index, country_pct.values, color='coral', edgecolor='white')
plt.title('Κατανομή Εσόδων ανά Χώρα — Top 10 (%)', fontsize=14)
plt.xlabel('Χώρα'); plt.ylabel('Ποσοστό Εσόδων (%)')
plt.xticks(rotation=45)
for bar, val in zip(bars, country_pct.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{val}%', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('diagram_4_2_country_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.2 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 9: EDA — Έσοδα ανά Ημέρα Εβδομάδας (Διάγραμμα 4.3)
# ============================================================
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
weekly = df.groupby('DayOfWeek')['Revenue'].sum().reindex(order)

plt.figure(figsize=(10,5))
bars = plt.bar(weekly.index, weekly.values, color='seagreen', edgecolor='white')
plt.title('Έσοδα ανά Ημέρα Εβδομάδας', fontsize=14)
plt.xlabel('Ημέρα'); plt.ylabel('Έσοδα (£)')
for bar, val in zip(bars, weekly.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20000, f'£{val/1e6:.2f}M', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('diagram_4_3_weekly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.3 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 10: EDA — Εποχικότητα (Διάγραμμα 4.4)
# ============================================================
df['Month'] = df['InvoiceDate'].dt.month
seasonal = df.groupby('Month')['Revenue'].sum()
month_names = ['Ιαν','Φεβ','Μαρ','Απρ','Μάι','Ιουν','Ιουλ','Αυγ','Σεπ','Οκτ','Νοε','Δεκ']

plt.figure(figsize=(12,5))
bars = plt.bar(range(1,13), seasonal.values, color='mediumpurple', edgecolor='white')
plt.title('Εποχικότητα Πωλήσεων ανά Μήνα', fontsize=14)
plt.xlabel('Μήνας'); plt.ylabel('Έσοδα (£)')
plt.xticks(range(1,13), month_names)
for bar, val in zip(bars, seasonal.values):
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10000, f'£{val/1e6:.2f}M', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('diagram_4_4_seasonality.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.4 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 11: Υπολογισμός RFM Features & Future CLV
# ============================================================
snapshot_date = df_feat['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df_feat.groupby('Customer ID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('Invoice',     'nunique'),
    Monetary  = ('Revenue',     'sum')
).reset_index()

future_clv = df_targ.groupby('Customer ID')['Revenue'].sum().reset_index()
future_clv.columns = ['Customer ID', 'Future_CLV']

df_model = rfm.merge(future_clv, on='Customer ID', how='left')
df_model['Future_CLV']     = df_model['Future_CLV'].fillna(0)
df_model['Monetary_log']   = np.log1p(df_model['Monetary'])
df_model['Future_CLV_log'] = np.log1p(df_model['Future_CLV'])

print(f'Πελάτες στο μοντέλο: {len(df_model):,}')
print(f'Με Future_CLV > 0  : {(df_model["Future_CLV"] > 0).sum():,}')
print(f'Με Future_CLV = 0  : {(df_model["Future_CLV"] == 0).sum():,}')
print()
print('=== Πίνακας 4.2: Περιγραφικά Στατιστικά RFM ===')
print(df_model[['Recency','Frequency','Monetary','Future_CLV']].describe().round(2))

In [ ]:
# ============================================================
# ΚΕΛΙ 12: Κατανομές RFM (Διάγραμμα 4.5)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Κατανομές Μεταβλητών RFM', fontsize=14, fontweight='bold')

axes[0].hist(df_model['Recency'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Κατανομή Recency')
axes[0].set_xlabel('Ημέρες από τελευταία αγορά')
axes[0].set_ylabel('Αριθμός Πελατών')

axes[1].hist(df_model['Frequency'], bins=40, color='coral', edgecolor='white')
axes[1].set_title('Κατανομή Frequency')
axes[1].set_xlabel('Αριθμός Συναλλαγών')
axes[1].set_ylabel('Αριθμός Πελατών')

axes[2].hist(df_model['Monetary'], bins=40, color='seagreen', edgecolor='white')
axes[2].set_title('Κατανομή Monetary')
axes[2].set_xlabel('Συνολικά Έσοδα (£)')
axes[2].set_ylabel('Αριθμός Πελατών')

plt.tight_layout()
plt.savefig('diagram_4_5_rfm_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.5 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 13: Correlation Matrix
# ============================================================
corr = df_model[['Recency','Frequency','Monetary','Future_CLV']].corr()

plt.figure(figsize=(7,5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.5)
plt.title('Πίνακας Συσχέτισης Pearson: RFM & Future CLV')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('Συσχέτιση με Future_CLV:')
print(corr['Future_CLV'].drop('Future_CLV').sort_values(ascending=False).round(3))

In [ ]:
# ============================================================
# ΚΕΛΙ 14: Clustering — Elbow & Scatter (Διάγραμμα 4.6)
# ============================================================
scaler_cl = StandardScaler()
rfm_scaled = scaler_cl.fit_transform(df_model[['Recency','Frequency','Monetary']])

inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

kmeans = KMeans(n_clusters=4, random_state=RANDOM_STATE, n_init=10)
df_model['Cluster_KM'] = kmeans.fit_predict(rfm_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(2,11), inertias, 'bo-', linewidth=2, markersize=8)
axes[0].axvline(x=4, color='red', linestyle='--', label='Επιλεγμένο k=4')
axes[0].set_title('Elbow Method — Επιλογή Αριθμού Clusters')
axes[0].set_xlabel('Αριθμός Clusters (k)')
axes[0].set_ylabel('Inertia'); axes[0].legend()

colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0']
for cluster in range(4):
    mask = df_model['Cluster_KM'] == cluster
    axes[1].scatter(df_model[mask]['Frequency'], df_model[mask]['Monetary'],
                    c=colors[cluster], label=f'Cluster {cluster}', alpha=0.5, s=20)
axes[1].set_title('Τμηματοποίηση (Frequency vs Monetary)')
axes[1].set_xlabel('Frequency'); axes[1].set_ylabel('Monetary (£)')
axes[1].legend()

plt.tight_layout()
plt.savefig('diagram_4_6_elbow_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.6 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 15: Clustering — Αξιολόγηση (Πίνακες 4.3 & 4.4)
# ============================================================
km_sil = silhouette_score(rfm_scaled, df_model['Cluster_KM'])
km_ch  = calinski_harabasz_score(rfm_scaled, df_model['Cluster_KM'])
km_db  = davies_bouldin_score(rfm_scaled, df_model['Cluster_KM'])

gmm = GaussianMixture(n_components=4, random_state=RANDOM_STATE)
df_model['Cluster_GMM'] = gmm.fit_predict(rfm_scaled)
gmm_sil = silhouette_score(rfm_scaled, df_model['Cluster_GMM'])
gmm_ch  = calinski_harabasz_score(rfm_scaled, df_model['Cluster_GMM'])
gmm_db  = davies_bouldin_score(rfm_scaled, df_model['Cluster_GMM'])

agg = AgglomerativeClustering(n_clusters=4)
df_model['Cluster_AGG'] = agg.fit_predict(rfm_scaled)
agg_sil = silhouette_score(rfm_scaled, df_model['Cluster_AGG'])
agg_ch  = calinski_harabasz_score(rfm_scaled, df_model['Cluster_AGG'])
agg_db  = davies_bouldin_score(rfm_scaled, df_model['Cluster_AGG'])

print('=== Πίνακας 4.3: Συγκριτική Αξιολόγηση Clustering ===')
print(f'{"Μοντέλο":<20} {"Silhouette":>12} {"Calinski-H":>12} {"Davies-B":>12}')
print(f'{"K-Means":<20} {km_sil:>12.4f} {km_ch:>12.2f} {km_db:>12.4f}')
print(f'{"GMM":<20} {gmm_sil:>12.4f} {gmm_ch:>12.2f} {gmm_db:>12.4f}')
print(f'{"Agglomerative":<20} {agg_sil:>12.4f} {agg_ch:>12.2f} {agg_db:>12.4f}')
print()
print('=== Πίνακας 4.4: Μέσες τιμές RFM ανά Cluster (K-Means) ===')
cluster_summary = df_model.groupby('Cluster_KM').agg(
    Recency=('Recency','mean'), Frequency=('Frequency','mean'),
    Monetary=('Monetary','mean'), Count=('Customer ID','count')
).round(2)
print(cluster_summary)

In [ ]:
# ============================================================
# ΚΕΛΙ 16: Clustering — Heatmap & Bar (Διάγραμμα 4.7)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cluster_summary[['Recency','Frequency','Monetary']],
            annot=True, cmap='YlGnBu', fmt='.2f', ax=axes[0])
axes[0].set_title('RFM Χαρακτηριστικά ανά Cluster')

cluster_summary['Count'].plot(kind='bar', ax=axes[1],
    color=['#2196F3','#4CAF50','#FF9800','#9C27B0'])
axes[1].set_title('Κατανομή Πελατών ανά Cluster')
axes[1].set_xlabel('Cluster'); axes[1].set_ylabel('Αριθμός Πελατών')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('diagram_4_7_clustering_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.7 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 17: Cluster Profiles (Διαγράμματα 4.8-4.11)
# ============================================================
clusters_info = [
    {'title':'Cluster 0 — Regular Customers','color':'#2196F3','light':'#E3F2FD',
     'count':'2.546 πελάτες (51,6%)','profile':'Ενεργοί με τακτικές αγορές και μέτρια δαπάνη.',
     'strategy':['Προγράμματα πιστότητας','Cross-selling προτάσεις','Newsletter με προσφορές'],
     'kpi':'Στόχος: Αύξηση Frequency & Monetary'},
    {'title':'Cluster 1 — Lost Customers','color':'#F44336','light':'#FFEBEE',
     'count':'2.348 πελάτες (47,6%)','profile':'Αδρανείς με μεγάλο χρονικό διάστημα από τελευταία αγορά.',
     'strategy':['Καμπάνιες επανενεργοποίησης','Εκπτωτικές προσφορές','Email με "Σας έχουμε χάσει"'],
     'kpi':'Στόχος: Μείωση Recency'},
    {'title':'Cluster 2 — High Value Customers','color':'#FF9800','light':'#FFF3E0',
     'count':'35 πελάτες (0,7%)','profile':'Εξαιρετικά ενεργοί με πολύ υψηλή δαπάνη.',
     'strategy':['VIP εξυπηρέτηση','Αποκλειστικές προσφορές','Προσωπικός account manager'],
     'kpi':'Στόχος: Διατήρηση & Αναβάθμιση'},
    {'title':'Cluster 3 — VIP Customers','color':'#9C27B0','light':'#F3E5F5',
     'count':'4 πελάτες (0,08%)','profile':'Στρατηγικοί με εξαιρετικά υψηλή αξία.',
     'strategy':['Άμεση επικοινωνία με διοίκηση','Προνομιακοί όροι συνεργασίας','Εξατομικευμένα πακέτα'],
     'kpi':'Στόχος: Διατήρηση πάση θυσία'}
]

for i, cluster in enumerate(clusters_info):
    fig, ax = plt.subplots(figsize=(12, 4))
    fig.patch.set_facecolor(cluster['light'])
    ax.set_facecolor(cluster['light'])
    ax.set_xlim(0,10); ax.set_ylim(0,10); ax.axis('off')
    ax.add_patch(FancyBboxPatch((0,8.3),10,1.5,boxstyle='round,pad=0.1',
                                facecolor=cluster['color'],edgecolor='none'))
    ax.text(5,9.1,cluster['title'],ha='center',va='center',fontsize=13,fontweight='bold',color='white')
    ax.text(5,7.9,cluster['count'],ha='center',va='center',fontsize=10,color=cluster['color'],fontweight='bold')
    ax.text(0.3,7.3,'Προφίλ:',fontsize=9,fontweight='bold')
    ax.text(0.3,6.7,cluster['profile'],fontsize=9,color='#555555')
    ax.text(5.3,7.3,'Στρατηγική:',fontsize=9,fontweight='bold')
    for j, action in enumerate(cluster['strategy']):
        ax.text(5.3,6.7-j*0.6,f'✔ {action}',fontsize=9,color='#555555')
    ax.add_patch(FancyBboxPatch((0.2,0.2),9.6,0.7,boxstyle='round,pad=0.1',
                                facecolor=cluster['color'],alpha=0.15,edgecolor=cluster['color']))
    ax.text(5,0.55,cluster['kpi'],ha='center',va='center',fontsize=9,color=cluster['color'],fontweight='bold')
    plt.tight_layout()
    filename = f'diagram_4_{8+i}_cluster_{i}_profile.png'
    plt.savefig(filename,dpi=150,bbox_inches='tight',facecolor=cluster['light'])
    plt.show()
    print(f'Διάγραμμα 4.{8+i} αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 18: BG/NBD + Gamma-Gamma (Davidson-Pilon, 2019)
#
# ΣΗΜΕΙΩΣΗ: Ο όρος 'frequency' στο BG/NBD διαφέρει από το
# RFM-Frequency του Πίνακα 4.2. Στο BG/NBD, η frequency
# ορίζεται ως ο αριθμός ΕΠΑΝΑΛΑΜΒΑΝΟΜΕΝΩΝ αγορών εξαιρουμένης
# της πρώτης. Πελάτες με μόνο 1 αγορά έχουν frequency=0 και
# εξαιρούνται από το Gamma-Gamma. Το RFM-Frequency μετρά το
# ΣΥΝΟΛΟ των μοναδικών τιμολογίων συμπεριλαμβανομένης της
# πρώτης αγοράς.
# ============================================================
observation_date = df_feat['InvoiceDate'].max()
summary = summary_data_from_transaction_data(
    df_feat, customer_id_col='Customer ID',
    datetime_col='InvoiceDate', monetary_value_col='Revenue',
    observation_period_end=observation_date
)
summary_gg = summary[summary['frequency'] > 0].copy()

print(f'Πελάτες BG/NBD              : {len(summary):,}')
print(f'Πελάτες Gamma-Gamma (freq>0): {len(summary_gg):,}')
print('(Υποσημείωση: frequency=0 στο BG/NBD = μόνο 1 αγορά — διαφορετικός ορισμός από RFM-Frequency)')

bgf = BetaGeoFitter(penalizer_coef=0.0)
bgf.fit(summary['frequency'], summary['recency'], summary['T'])
print(f'\nBG/NBD παράμετροι: a={bgf.params_["a"]:.4f}, b={bgf.params_["b"]:.4f}, alpha={bgf.params_["alpha"]:.4f}, r={bgf.params_["r"]:.4f}')

ggf = GammaGammaFitter(penalizer_coef=0.0)
ggf.fit(summary_gg['frequency'], summary_gg['monetary_value'])
print(f'Gamma-Gamma παράμετροι: p={ggf.params_["p"]:.4f}, q={ggf.params_["q"]:.4f}, v={ggf.params_["v"]:.4f}')

clv_bgf = ggf.customer_lifetime_value(
    bgf, summary_gg['frequency'], summary_gg['recency'],
    summary_gg['T'], summary_gg['monetary_value'],
    time=6, discount_rate=0.01
)
clv_bgf_df = clv_bgf.reset_index()
clv_bgf_df.columns = ['Customer ID', 'CLV_probabilistic']

eval_bgf = df_model.merge(clv_bgf_df, on='Customer ID', how='inner')
y_true_bgf = eval_bgf['Future_CLV']
y_pred_bgf = eval_bgf['CLV_probabilistic']

mae_bgf   = mean_absolute_error(y_true_bgf, y_pred_bgf)
rmse_bgf  = np.sqrt(mean_squared_error(y_true_bgf, y_pred_bgf))
mape_bgf  = np.mean(np.abs((y_true_bgf-y_pred_bgf)/(y_true_bgf+1)))*100
r2_bgf    = r2_score(y_true_bgf, y_pred_bgf)
sp_bgf, _ = spearmanr(y_true_bgf, y_pred_bgf)

print('\n=== BG/NBD + Gamma-Gamma ===')
print(f'  MAE: {mae_bgf:.2f} £ | RMSE: {rmse_bgf:.2f} £ | MAPE: {mape_bgf:.2f}%')
print(f'  R²: {r2_bgf:.4f} | Spearman: {sp_bgf:.4f}')
print('  ΣΗΜΕΙΩΣΗ: Το R² του BG/NBD ΔΕΝ είναι άμεσα συγκρίσιμο με των ML μοντέλων')
print('  λόγω διαφορετικής μεθοδολογίας υπολογισμού CLV. Παρατίθεται για πληρότητα.')

bgf_results = {'Model':'BG/NBD + Gamma-Gamma','MAE':round(mae_bgf,2),
               'RMSE':round(rmse_bgf,2),'MAPE':round(mape_bgf,2),
               'R2':round(r2_bgf,4),'Spearman':round(sp_bgf,4),'CV_R2':'N/A'}

In [ ]:
# ============================================================
# ΚΕΛΙ 19: ML Setup
#
# TimeSeriesSplit: Οι πελάτες ταξινομούνται βάσει ημερομηνίας
# πρώτης αγοράς ώστε να διατηρείται η χρονική σειρά κατά την
# cross-validation. Κάθε fold εκπαιδεύεται σε παλαιότερα
# δεδομένα και αξιολογείται σε νεότερα.
#
# StandardScaler: fit ΜΟΝΟ στο training set, transform και
# στο training και στο test set — αποφυγή data leakage.
# ============================================================
FEATURES = ['Recency','Frequency','Monetary_log']
X = df_model[FEATURES].values
y_log = df_model['Future_CLV_log'].values
y_raw = df_model['Future_CLV'].values

# Ταξινόμηση κατά ημερομηνία πρώτης αγοράς για TimeSeriesSplit
first_purchase = df_feat.groupby('Customer ID')['InvoiceDate'].min().reset_index()
first_purchase.columns = ['Customer ID','First_Purchase']
df_model_sorted = df_model.merge(first_purchase, on='Customer ID', how='left')
df_model_sorted = df_model_sorted.sort_values('First_Purchase').reset_index(drop=True)

X_sorted = df_model_sorted[FEATURES].values
y_log_sorted = df_model_sorted['Future_CLV_log'].values
y_raw_sorted = df_model_sorted['Future_CLV'].values

# Train/Test split 80/20
X_train, X_test, y_train_log, y_test_log, y_train_raw, y_test_raw = train_test_split(
    X, y_log, y_raw, test_size=0.20, random_state=RANDOM_STATE
)

# StandardScaler — fit ΜΟΝΟ στο train
scaler_ml = StandardScaler()
X_train_scaled = scaler_ml.fit_transform(X_train)  # fit + transform στο train
X_test_scaled  = scaler_ml.transform(X_test)        # ΜΟΝΟ transform στο test

# 5-fold TimeSeriesSplit με ταξινόμηση κατά πρώτη αγορά
tscv = TimeSeriesSplit(n_splits=5)

def evaluate_model(name, y_true_raw, y_pred_log):
    y_pred_raw = np.maximum(np.expm1(y_pred_log), 0)
    mae  = mean_absolute_error(y_true_raw, y_pred_raw)
    rmse = np.sqrt(mean_squared_error(y_true_raw, y_pred_raw))
    mape = np.mean(np.abs((y_true_raw-y_pred_raw)/(y_true_raw+1)))*100
    r2   = r2_score(y_true_raw, y_pred_raw)
    sp, _ = spearmanr(y_true_raw, y_pred_raw)
    print(f'\n=== {name} ===')
    print(f'  MAE: {mae:.2f} £ | RMSE: {rmse:.2f} £ | MAPE: {mape:.2f}%')
    print(f'  R²: {r2:.4f} | Spearman: {sp:.4f}')
    return {'Model':name,'MAE':round(mae,2),'RMSE':round(rmse,2),
            'MAPE':round(mape,2),'R2':round(r2,4),'Spearman':round(sp,4)}, y_pred_raw

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
print('Cross-validation: 5-fold TimeSeriesSplit (ταξινόμηση κατά πρώτη αγορά)')
print('StandardScaler: fit μόνο στο train set')

In [ ]:
# ============================================================
# ΚΕΛΙ 20: Random Forest (Pedregosa et al., 2011)
# ============================================================
rf = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_split=10,
                            min_samples_leaf=5, max_features='sqrt',
                            random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train_log)
rf_metrics, rf_pred = evaluate_model('Random Forest', y_test_raw, rf.predict(X_test))
cv_rf = cross_val_score(rf, X_sorted, y_log_sorted, cv=tscv, scoring='r2', n_jobs=-1)
print(f'  CV R² (5-fold TS): {cv_rf.mean():.4f} ± {cv_rf.std():.4f}')
rf_metrics['CV_R2'] = f'{cv_rf.mean():.4f} ± {cv_rf.std():.4f}'

In [ ]:
# ============================================================
# ΚΕΛΙ 21: XGBoost (Chen & Guestrin, 2016)
# ============================================================
xgb_model = xgb.XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                               subsample=0.8, colsample_bytree=0.8, min_child_weight=10,
                               reg_alpha=0.1, reg_lambda=1.0,
                               random_state=RANDOM_STATE, verbosity=0, n_jobs=-1)
xgb_model.fit(X_train, y_train_log)
xgb_metrics, xgb_pred = evaluate_model('XGBoost', y_test_raw, xgb_model.predict(X_test))
cv_xgb = cross_val_score(xgb_model, X_sorted, y_log_sorted, cv=tscv, scoring='r2', n_jobs=-1)
print(f'  CV R² (5-fold TS): {cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}')
xgb_metrics['CV_R2'] = f'{cv_xgb.mean():.4f} ± {cv_xgb.std():.4f}'

In [ ]:
# ============================================================
# ΚΕΛΙ 22: LightGBM (Ke et al., 2017)
# ============================================================
lgbm_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6,
                                 num_leaves=31, subsample=0.8, colsample_bytree=0.8,
                                 min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
                                 random_state=RANDOM_STATE, verbose=-1)
lgbm_model.fit(X_train, y_train_log)
lgbm_metrics, lgbm_pred = evaluate_model('LightGBM', y_test_raw, lgbm_model.predict(X_test))
cv_lgbm = cross_val_score(lgbm_model, X_sorted, y_log_sorted, cv=tscv, scoring='r2', n_jobs=-1)
print(f'  CV R² (5-fold TS): {cv_lgbm.mean():.4f} ± {cv_lgbm.std():.4f}')
lgbm_metrics['CV_R2'] = f'{cv_lgbm.mean():.4f} ± {cv_lgbm.std():.4f}'

In [ ]:
# ============================================================
# ΚΕΛΙ 23: Gradient Boosting (Pedregosa et al., 2011)
# ============================================================
gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=5,
                                 subsample=0.8, min_samples_leaf=10, random_state=RANDOM_STATE)
gbr.fit(X_train, y_train_log)
gbr_metrics, gbr_pred = evaluate_model('Gradient Boosting', y_test_raw, gbr.predict(X_test))
cv_gbr = cross_val_score(gbr, X_sorted, y_log_sorted, cv=tscv, scoring='r2', n_jobs=-1)
print(f'  CV R² (5-fold TS): {cv_gbr.mean():.4f} ± {cv_gbr.std():.4f}')
gbr_metrics['CV_R2'] = f'{cv_gbr.mean():.4f} ± {cv_gbr.std():.4f}'

In [ ]:
# ============================================================
# ΚΕΛΙ 24: Neural Network (Abadi et al., 2016)
# Αρχιτεκτονική: 3 κρυφά επίπεδα (64-32-16), ReLU, Dropout=0.2
# Εκπαίδευση: Adam, MSE, batch=32, epochs=100
# Είσοδος: StandardScaler (fit μόνο στο train)
# ============================================================
model_nn = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(3,)),
    layers.Dropout(0.2),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])
model_nn.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model_nn.fit(X_train_scaled, y_train_log, epochs=100, batch_size=32,
                       validation_split=0.2, verbose=0)

nn_pred_log = model_nn.predict(X_test_scaled).flatten()
nn_metrics, nn_pred = evaluate_model('Neural Network', y_test_raw, nn_pred_log)
nn_metrics['CV_R2'] = 'N/A'

# Learning Curve (Διάγραμμα 4.14)
plt.figure(figsize=(10,4))
plt.plot(history.history['loss'], label='Training Loss', color='steelblue')
plt.plot(history.history['val_loss'], label='Validation Loss', color='coral')
plt.title('Neural Network — Learning Curve (Training vs Validation Loss)')
plt.xlabel('Epoch'); plt.ylabel('Loss (MSE)'); plt.legend()
plt.tight_layout()
plt.savefig('diagram_4_14_nn_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.14 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 25: Πίνακας 4.5 & Διάγραμμα 4.12
# ============================================================
all_results = [bgf_results, rf_metrics, xgb_metrics, lgbm_metrics, gbr_metrics, nn_metrics]
results_df = pd.DataFrame(all_results)

print('=' * 80)
print('ΠΙΝΑΚΑΣ 4.5: ΣΥΓΚΡΙΤΙΚΗ ΑΞΙΟΛΟΓΗΣΗ ΜΟΝΤΕΛΩΝ CLV')
print('=' * 80)
print(results_df[['Model','MAE','RMSE','MAPE','R2','Spearman','CV_R2']].to_string(index=False))
print('=' * 80)
print('Σημείωση: Το R² του BG/NBD + Gamma-Gamma δεν είναι άμεσα συγκρίσιμο με')
print('των ML μοντέλων λόγω διαφορετικής μεθοδολογίας. Παρατίθεται για πληρότητα.')
print('Υψηλότερο Spearman μεταξύ ML μοντέλων: Neural Network (0.6329)')

results_df.to_csv('model_results.csv', index=False)

# Διάγραμμα 4.12
ml_df = results_df[results_df['Model'] != 'BG/NBD + Gamma-Gamma'].copy()
colors5 = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Συγκριτική Αξιολόγηση ML Μοντέλων CLV', fontsize=13, fontweight='bold')

axes[0].bar(ml_df['Model'], ml_df['R2'].astype(float), color=colors5)
axes[0].set_title('R² (υψηλότερο = καλύτερο)')
axes[0].set_ylabel('R²'); axes[0].tick_params(axis='x', rotation=25)
for i, v in enumerate(ml_df['R2'].astype(float)):
    axes[0].text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=8)

axes[1].bar(ml_df['Model'], ml_df['RMSE'].astype(float), color=colors5)
axes[1].set_title('RMSE (χαμηλότερο = καλύτερο)')
axes[1].set_ylabel('RMSE (£)'); axes[1].tick_params(axis='x', rotation=25)

axes[2].bar(ml_df['Model'], ml_df['Spearman'].astype(float), color=colors5)
axes[2].set_title('Spearman (υψηλότερο = καλύτερο)')
axes[2].set_ylabel('Spearman r'); axes[2].tick_params(axis='x', rotation=25)
for i, v in enumerate(ml_df['Spearman'].astype(float)):
    axes[2].text(i, v+0.002, f'{v:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('diagram_4_12_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.12 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 26: Predicted vs Actual (Διάγραμμα 4.13)
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle('Predicted vs Actual Future CLV — ML Models', fontsize=14, fontweight='bold')

models_info = [
    ('Random Forest', rf_pred, '#2196F3'),
    ('XGBoost', xgb_pred, '#4CAF50'),
    ('LightGBM', lgbm_pred, '#FF9800'),
    ('Gradient Boosting', gbr_pred, '#9C27B0')
]
for ax, (name, y_pred, color) in zip(axes.flat, models_info):
    ax.scatter(y_test_raw, y_pred, alpha=0.25, s=8, color=color)
    max_val = max(y_test_raw.max(), y_pred.max())
    ax.plot([0,max_val],[0,max_val],'r--',linewidth=1.5,label='Τέλεια πρόβλεψη')
    ax.set_xlabel('Actual Future CLV (£)'); ax.set_ylabel('Predicted CLV (£)')
    ax.set_title(name); ax.legend()

plt.tight_layout()
plt.savefig('diagram_4_13_predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.13 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 27: Feature Importance (Διάγραμμα 4.15)
# ============================================================
imp_df = pd.DataFrame({
    'Feature': FEATURES,
    'Random Forest': rf.feature_importances_,
    'XGBoost': xgb_model.feature_importances_,
    'LightGBM': lgbm_model.feature_importances_/lgbm_model.feature_importances_.sum(),
    'Gradient Boosting': gbr.feature_importances_
})
imp_melted = imp_df.melt(id_vars='Feature', var_name='Model', value_name='Importance')

plt.figure(figsize=(10,5))
sns.barplot(data=imp_melted, x='Feature', y='Importance', hue='Model')
plt.title('Feature Importance ανά Μοντέλο Μηχανικής Μάθησης')
plt.ylabel('Importance (normalized)')
plt.tight_layout()
plt.savefig('diagram_4_15_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(imp_df.round(4).to_string(index=False))
print('Διάγραμμα 4.15 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 28: SHAP Analysis — Bar Plot (Διάγραμμα 4.16)
# Εφαρμόζεται στο Gradient Boosting (υψηλότερο R²)
# Lundberg & Lee (2017)
# ============================================================
explainer = shap.TreeExplainer(gbr)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(8,4))
shap.summary_plot(shap_values, X_test, feature_names=FEATURES, plot_type='bar', show=False)
plt.title('SHAP Global Feature Importance — Gradient Boosting')
plt.tight_layout()
plt.savefig('diagram_4_16_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.16 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 29: SHAP Beeswarm Plot (Διάγραμμα 4.17)
# ============================================================
plt.figure(figsize=(8,4))
shap.summary_plot(shap_values, X_test, feature_names=FEATURES, show=False)
plt.title('SHAP Beeswarm Plot — Gradient Boosting')
plt.tight_layout()
plt.savefig('diagram_4_17_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.17 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 30: SHAP Dependence Plots (Διάγραμμα 4.18)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('SHAP Dependence Plots — Gradient Boosting', fontsize=13)
for i, feat in enumerate(FEATURES):
    shap.dependence_plot(i, shap_values, X_test, feature_names=FEATURES, ax=axes[i], show=False)
plt.tight_layout()
plt.savefig('diagram_4_18_shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.18 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 31: Residual Analysis (Διάγραμμα 4.19)
# ============================================================
residuals = y_test_raw - gbr_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Residual Analysis — Gradient Boosting', fontsize=13)

axes[0].scatter(gbr_pred, residuals, alpha=0.3, s=8, color='steelblue')
axes[0].axhline(y=0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted CLV (£)'); axes[0].set_ylabel('Residuals (£)')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[1].axvline(x=0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (£)'); axes[1].set_ylabel('Συχνότητα')
axes[1].set_title('Κατανομή Residuals')

plt.tight_layout()
plt.savefig('diagram_4_19_residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Διάγραμμα 4.19 αποθηκεύτηκε')

In [ ]:
# ============================================================
# ΚΕΛΙ 32: Πίνακας 4.6 — Top 10 Πελάτες
# ============================================================
X_full = df_model[FEATURES].values
df_model['CLV_RF']       = np.maximum(np.expm1(rf.predict(X_full)), 0).round(2)
df_model['CLV_XGB']      = np.maximum(np.expm1(xgb_model.predict(X_full)), 0).round(2)
df_model['CLV_LGBM']     = np.maximum(np.expm1(lgbm_model.predict(X_full)), 0).round(2)
df_model['CLV_GBR']      = np.maximum(np.expm1(gbr.predict(X_full)), 0).round(2)
df_model['CLV_Ensemble'] = df_model[['CLV_RF','CLV_XGB','CLV_LGBM','CLV_GBR']].mean(axis=1).round(2)

print('=== Πίνακας 4.6: Top 10 Πελάτες βάσει Ensemble CLV ===')
print(df_model.nlargest(10,'CLV_Ensemble')[
    ['Customer ID','Recency','Frequency','Monetary','Future_CLV','CLV_Ensemble']
].to_string(index=False))

df_model.to_csv('clv_all_predictions.csv', index=False)
print('\nΑποθηκεύτηκε: clv_all_predictions.csv')

In [ ]:
# ============================================================
# ΚΕΛΙ 33: Κατέβασμα Όλων των Αποτελεσμάτων
# ============================================================
from google.colab import files

output_files = [
    'diagram_4_1_monthly_revenue.png',
    'diagram_4_2_country_revenue.png',
    'diagram_4_3_weekly_revenue.png',
    'diagram_4_4_seasonality.png',
    'diagram_4_5_rfm_distributions.png',
    'diagram_4_6_elbow_clusters.png',
    'diagram_4_7_clustering_results.png',
    'diagram_4_8_cluster_0_profile.png',
    'diagram_4_9_cluster_1_profile.png',
    'diagram_4_10_cluster_2_profile.png',
    'diagram_4_11_cluster_3_profile.png',
    'diagram_4_12_model_comparison.png',
    'diagram_4_13_predicted_vs_actual.png',
    'diagram_4_14_nn_learning_curve.png',
    'diagram_4_15_feature_importance.png',
    'diagram_4_16_shap_bar.png',
    'diagram_4_17_shap_beeswarm.png',
    'diagram_4_18_shap_dependence.png',
    'diagram_4_19_residual_analysis.png',
    'model_results.csv',
    'clv_all_predictions.csv'
]

for f in output_files:
    try:
        files.download(f)
        print(f'✅ {f}')
    except:
        print(f'⚠️ {f} — δεν βρέθηκε')

print('\n✅ ΟΛΟΚΛΗΡΩΘΗΚΕ Η ΑΝΑΛΥΣΗ')
print('Αναπαραγωγιμότητα: random_state=42 | TimeSeriesSplit(5) | StandardScaler fit μόνο στο train')